## 1. Before You Begin

**How do we turn a flood of messy messages into typed decisions our code can act on?** In this notebook we work the Contoso Music Festival ops desk, the festival story from the [capsule README](README.md). We send one message and ask for a team, a severity, and a danger check in a single request. Then we run a small batch, break it with messages that fit no team, and fix it.

![Site map of the fictional Contoso Music Festival](assets/contoso-music-festival-map.png)

| Detail | Value |
|---|---|
| Model card | [Microsoft-Decision-1 — Foundry catalog](https://ai.azure.com/catalog/models/Microsoft-Decision-1) |
| Docs | [Deploy and use Microsoft-Decision-1](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision) |
| Pricing | See the Pricing tab on the model card |
| Released | 2026-10-09 |
| Background | [Decision models primer](../../../docs/primers/decision-models.md) |

**Prerequisites**

1. A Microsoft Foundry project with Microsoft-Decision-1 deployed. See [models/quickstart/](../../quickstart/README.md) for first-time setup, then run `./scripts/3-deploy-model.sh decision-1 Microsoft-Decision-1 --env-var AZURE_DECISION_1_DEPLOYMENT`.
2. The environment variables listed in section 2.
3. `requests`, `python-dotenv`, and `pandas` installed. The next cell installs them.

## 2. Verify your environment

The cell below checks the three variables this notebook needs and builds the decision endpoint, `<resource>/providers/microsoft/v1/systemone`. It makes no paid request.

In [1]:
%pip install requests python-dotenv pandas matplotlib --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

import os
from urllib.parse import urlparse

# Your .env file needs these variables:
#   MICROSOFT_FOUNDRY_ENDPOINT=https://<resource>.services.ai.azure.com/api/projects/<project>
#   MICROSOFT_FOUNDRY_API_KEY=<your key>
#   AZURE_DECISION_1_DEPLOYMENT=<your Microsoft-Decision-1 deployment name>
REQUIRED = {
    "MICROSOFT_FOUNDRY_ENDPOINT": "Foundry project endpoint",
    "MICROSOFT_FOUNDRY_API_KEY": "Foundry API key",
    "AZURE_DECISION_1_DEPLOYMENT": "Microsoft-Decision-1 deployment name",
}
missing = [name for name in REQUIRED if not os.environ.get(name)]
if missing:
    raise EnvironmentError(
        f"Set these in your .env file before continuing: {missing}. "
        "See models/quickstart/README.md, then deploy the model with "
        "./scripts/3-deploy-model.sh decision-1 Microsoft-Decision-1 --env-var AZURE_DECISION_1_DEPLOYMENT"
    )

parsed = urlparse(os.environ["MICROSOFT_FOUNDRY_ENDPOINT"])
DECISION_URL = f"{parsed.scheme}://{parsed.netloc}/providers/microsoft/v1/systemone"
API_KEY = os.environ["MICROSOFT_FOUNDRY_API_KEY"]
DEPLOYMENT = os.environ["AZURE_DECISION_1_DEPLOYMENT"]

print(f"Deployment : {DEPLOYMENT}")
print("Environment check passed.")

Deployment : Decision-1
Environment check passed.


## 3. Frame the question

It's 17:05 on Saturday. A guest writes, "My friend just collapsed near the barrier at the front." The desk needs four quick answers before anyone types a reply:

- **Team.** Which team gets this? (a choice)
- **Severity.** How serious is it? (a score on an ordered scale)
- **Danger.** Is someone in immediate danger? (a yes-or-no probability, which the API calls a noul)
- **Repeat.** Is it something we already know about? (another noul, using open incidents as context)

Microsoft-Decision-1 doesn't write a reply. For each question we define, it returns a typed answer with probabilities. By the end of this notebook we'll have a table of decisions for a batch of messages, and we'll know why a message that fits no team needs an explicit way out.

## 4. Configure the client

Every call is one `POST` with a `state` (the evidence, as text or JSON) and a set of named `questions`. The helper below sends the request, retries with backoff on `429 Too Many Requests`, and keeps the error details when something else goes wrong. `decide_many` runs several requests in parallel and returns the answers in the same order.

In [3]:
import json, time
import concurrent.futures
from pathlib import Path

import requests

ASSETS = Path("assets")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

MAX_WORKERS = 4   # a capacity-1 deployment returned 429 responses at eight workers


def decide(state, questions, retries=6):
    """POST one decision request and return the parsed JSON response."""
    body = {"model": DEPLOYMENT, "state": state, "questions": questions}
    for attempt in range(retries):
        response = requests.post(
            DECISION_URL,
            headers={"api-key": API_KEY, "Content-Type": "application/json"},
            json=body,
            timeout=60,
        )
        if response.status_code == 429:   # rate limited: wait, then try again
            time.sleep(float(response.headers.get("Retry-After", 0)) or 1.5 * 2 ** min(attempt, 4))
            continue
        if not response.ok:
            raise requests.HTTPError(f"{response.status_code} {response.reason}: {response.text}", response=response)
        return response.json()
    raise RuntimeError("Still rate limited after several retries. Lower MAX_WORKERS or raise the deployment capacity.")


def decide_many(jobs, workers=MAX_WORKERS):
    """Run (state, questions) jobs in parallel; responses come back in input order."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as executor:
        return list(executor.map(lambda job: decide(*job), jobs))


def bar(p, width=20):
    filled = round(p * width)
    return "█" * filled + "·" * (width - filled)

## 5. Send the first decision

We start small: one message and one choice question. A choice question needs a fixed set of options. Each option gets a short description so the model knows what belongs there.

In [4]:
MESSAGES = json.loads((ASSETS / "messages.json").read_text(encoding="utf-8"))
BY_ID = {m["id"]: m for m in MESSAGES}

TEAMS = {
    "medical": "Injuries, illness, allergic reactions, heat or cold exposure, and requests for the medical tent",
    "security": "Fights, theft, weapons, fire, suspicious items or people, missing children, and crowd-control problems",
    "facilities": "Site infrastructure: stage and rigging, power, lighting, water, toilets, trash, tents, signage, and showers",
    "transport": "Shuttles, buses, parking, rideshare pickup, bikes, and traffic around the site",
    "guest_services": "Tickets, wristbands, refunds, lost property, accessibility, charging lockers, general questions, and feedback",
    "vendors": "Food and merchandise stalls: prices, food quality, hygiene, supplies, and stall equipment",
    "command": "Weather and site-wide decisions: lightning, wind, evacuation, shelter, show holds, and opening or closing gates",
}


def make_state(message):
    """The evidence for one message, as JSON."""
    state = {"channel": message["channel"], "zone": message["zone"], "time": message["time"], "message": message["text"]}
    if message.get("open_incidents"):
        state["open_incidents"] = message["open_incidents"]
    return state


def team_question(teams):
    return {
        "type": "choice",
        "instructions": "Which team should handle this message?",
        "criteria": teams,
    }


collapse = BY_ID["m29"]
first = decide(make_state(collapse), {"team": team_question(TEAMS)})
print(json.dumps(first, indent=2))

{
  "model": "microsoft-decision-1",
  "answers": {
    "team": {
      "choice": "medical",
      "confidence": 0.9970496885387631,
      "probabilities": {
        "command": 0.0005516856306741977,
        "facilities": 0.00042965320117830703,
        "guest_services": 0.0002299767862304577,
        "medical": 0.9974711616046541,
        "security": 0.0007083783718070761,
        "transport": 0.0002299767862304577,
        "vendors": 0.00037916761922541196
      },
      "type": "choice"
    }
  },
  "usage": {
    "input_tokens": 231,
    "output_tokens": 1
  }
}


### 5.1 Read the response

Under `answers`, each question we named comes back with its own typed answer. For a choice question that includes:

- `choice`, the option with the highest probability
- `probabilities`, one value per option, adding up to about 1
- `confidence`, the model's confidence in the choice

`usage` reports the tokens spent. The request was short, and so was the answer, because the model returns numbers instead of prose.

## 6. Ask three questions in one request

Questions that share a state can share a request, which keeps the context consistent and saves round trips. We add a **score** question for severity and a **noul** question for danger. A score question takes levels ordered from lowest to highest, and the answer is the probability-weighted position on that scale.

In [5]:
SEVERITY_LEVELS = [
    "Routine: a question, feedback, or a minor annoyance",
    "Needs attention: a problem a team should fix soon",
    "Urgent: a safety risk or a fast-growing problem",
    "Emergency: someone may be seriously hurt right now",
]


def make_questions(teams, with_repeat=False):
    questions = {
        "team": team_question(teams),
        "severity": {
            "type": "score",
            "instructions": "How severe is the situation described?",
            "criteria": SEVERITY_LEVELS,
        },
        "danger": {
            "type": "noul",
            "instructions": "Does the message describe a person who may be seriously hurt or in immediate physical danger right now?",
            "criteria": {
                "true": "Someone is or may be in immediate physical danger",
                "false": "No one is described as being in immediate danger",
            },
        },
    }
    if with_repeat:
        questions["repeat"] = {
            "type": "noul",
            "instructions": "Does this message report the same problem as one of the open incidents?",
        }
    return questions


def show_card(message, response):
    """Print one decision as a small card."""
    answers = response["answers"]
    print(f'{message["time"]} · {message["channel"]} · {message["zone"]}')
    print(f'"{message["text"]}"\n')
    team = answers["team"]
    print(f"Team     {team['choice']}   (confidence {team['confidence']:.2f})")
    for option, p in sorted(team["probabilities"].items(), key=lambda item: -item[1])[:3]:
        print(f"           {option:<15}{bar(p)} {p:.2f}")
    severity = answers["severity"]
    top_level = max(severity["probabilities"], key=severity["probabilities"].get)
    print(f"\nSeverity {severity['score']:.2f} on a 0 to 3 scale, closest level: {severity['legend'][top_level].split(':')[0]}")
    print(f"Danger   {bar(answers['danger']['noul'])} {answers['danger']['noul']:.2f}")
    if "repeat" in answers:
        print(f"Repeat   {bar(answers['repeat']['noul'])} {answers['repeat']['noul']:.2f}")


response = decide(make_state(collapse), make_questions(TEAMS))
show_card(collapse, response)

17:05 · guest app · Main Stage
"My friend just collapsed near the barrier at the front. She's breathing but won't wake up!!"

Team     medical   (confidence 1.00)
           medical        ████████████████████ 1.00
           security       ···················· 0.00
           command        ···················· 0.00

Severity 3.00 on a 0 to 3 scale, closest level: Emergency
Danger   ████████████████████ 1.00


### 6.1 Use JSON as the state

The state doesn't have to be a sentence. Here the guest's message arrives with the desk's open incidents, and a fourth question asks whether it repeats one of them. We pass the open incidents in the state and the model reads them next to the message.

In [6]:
reported_twice = BY_ID["m46"]
print(json.dumps(make_state(reported_twice), indent=2, ensure_ascii=False))
print()

response = decide(make_state(reported_twice), make_questions(TEAMS, with_repeat=True))
show_card(reported_twice, response)

{
  "channel": "guest app",
  "zone": "Gate B",
  "time": "17:44",
  "message": "The fence at Gate B is down and people are pushing through. This is scary.",
  "open_incidents": [
    "Gate B: fence panel down, crowd coming through (reported 17:40, security on site)",
    "Shuttle Hub: city shuttle queue over 40 minutes (reported 17:30, transport adding buses)"
  ]
}



17:44 · guest app · Gate B
"The fence at Gate B is down and people are pushing through. This is scary."

Team     security   (confidence 0.93)
           security       ███████████████████· 0.94
           command        █··················· 0.05
           facilities     ···················· 0.00

Severity 2.18 on a 0 to 3 scale, closest level: Urgent
Danger   ████████████████···· 0.80
Repeat   ████████████████████ 1.00


The repeat probability sits at one end of the scale because the message and an open incident describe the same fence. When the desk already has an incident open, the new message can attach to it instead of opening a second one. Try `m48`, "Toilet queue is huge but at least they're clean here", which has open incidents in its state but isn't a repeat of any of them.

## 7. Decide a batch and read it as data

One card is easy to read. A shift has hundreds of messages, so we want the answers as a table. `read_decision` flattens one response into a row, and `run_batch` sends a list of messages and returns a DataFrame next to the labels we wrote for each message.

The starter batch has 13 messages. Eight are clear, and five are vague, such as "Help" and "It's happening again over there." We use the seven teams above and don't give the model a way out yet.

In [7]:
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 46)

STARTER = ["m02", "m11", "m17", "m29", "m35", "m38", "m40", "m46", "m27", "m34", "m43", "m56", "m69"]


def read_decision(response):
    answers = response["answers"]
    return {
        "team": answers["team"]["choice"],
        "conf": round(answers["team"]["confidence"], 2),
        "sev": round(answers["severity"]["score"], 2),
        "danger": round(answers["danger"]["noul"], 2),
    }


def run_batch(ids, teams):
    batch = [BY_ID[i] for i in ids]
    jobs = [(make_state(m), make_questions(teams, bool(m.get("open_incidents")))) for m in batch]
    rows = []
    for message, response in zip(batch, decide_many(jobs)):
        expected = message["expected"]
        row = {"id": message["id"], "message": message["text"][:44], "expected": expected["team"], **read_decision(response)}
        row["exp_sev"] = expected["severity"]
        row["match"] = row["team"] == row["expected"]
        rows.append(row)
    return pd.DataFrame(rows)


before = run_batch(STARTER, TEAMS)
before

,id,message,expected,team,conf,sev,danger,exp_sev,match
0,m02,Barcode on my ticket won't scan and there's,guest_services,guest_services,0.99,0.85,0.00,1.0,True
1,m11,Feeling dizzy and sick after a day in the su,medical,medical,1.00,1.65,0.32,1.0,True
2,m17,"Mi hija se perdió cerca de la entrada, tiene",security,security,1.00,2.96,0.92,3.0,True
3,m29,My friend just collapsed near the barrier at,medical,medical,1.00,3.00,1.00,3.0,True
4,m35,Left speaker tower is leaning. Looks like th,facilities,facilities,1.00,1.91,0.35,3.0,True
5,m38,Drunk guy shouting at vendors and knocking o,security,security,1.00,2.08,0.94,1.0,True
6,m40,Last shuttle to the city is full and we've w,transport,transport,0.99,1.04,0.00,1.0,True
7,m46,The fence at Gate B is down and people are p,security,security,0.93,2.18,0.75,3.0,True
8,m27,It's happening again over there.,unclear,security,0.92,1.33,0.04,NaN,False
9,m34,Help,unclear,security,0.62,2.63,0.95,NaN,False


## 8. Observe: where does it do well, and where doesn't it?

Compare the `team` column with `expected`. The clear messages should land on the right team with high confidence, and `sev` should rise with the seriousness of the message. Then look at the vague ones.

In [8]:
clear = before[before.expected != "unclear"]
vague = before[before.expected == "unclear"]

print(f"Clear messages: {clear.match.mean():.0%} routed to the expected team ({len(clear)} messages)")
print(f"Vague messages: {vague.match.mean():.0%} routed to the expected team ({len(vague)} messages)\n")
vague[["id", "message", "team", "conf"]]

Clear messages: 100% routed to the expected team (8 messages)
Vague messages: 0% routed to the expected team (5 messages)



,id,message,team,conf
8,m27,It's happening again over there.,security,0.92
9,m34,Help,security,0.62
10,m43,Can someone come?,guest_services,0.40
11,m56,Is it safe?,command,0.84
12,m69,Somebody should take a look at the thing nea,security,0.91


## 9. Diagnose: why did a vague message get a team?

Every vague message came back with a team, and some of those picks carry real confidence. That can look like a model problem, but the cause is in the question we wrote. A choice question picks exactly one option from the list we gave, and our list has no option for "this doesn't say enough." The model can only spread its probability across seven teams.

We can check that from the evidence. First, the probabilities behind each vague pick. Then the mean confidence for clear and vague messages side by side.

In [9]:
for _, row in vague.iterrows():
    message = BY_ID[row["id"]]
    answer = decide(make_state(message), {"team": team_question(TEAMS)})["answers"]["team"]
    top = sorted(answer["probabilities"].items(), key=lambda item: -item[1])[:3]
    print(f'{row["id"]}  "{message["text"]}"')
    print("      " + "   ".join(f"{name} {p:.2f}" for name, p in top))

summary = (
    before.assign(kind=before.expected.map(lambda team: "vague" if team == "unclear" else "clear"))
    .groupby("kind")
    .agg(messages=("id", "count"), mean_confidence=("conf", "mean"))
    .round(2)
)
print()
summary

m27  "It's happening again over there."
      security 0.94   command 0.03   facilities 0.02


m34  "Help"
      security 0.65   medical 0.19   guest_services 0.09


m43  "Can someone come?"
      guest_services 0.45   security 0.27   medical 0.19


m56  "Is it safe?"
      command 0.86   guest_services 0.06   security 0.04


m69  "Somebody should take a look at the thing near the stage."
      security 0.90   facilities 0.09   command 0.01



,messages,mean_confidence
kind,,
clear,8,0.99
vague,5,0.74


The model isn't being careless here, and it can't say "I don't know" because we never offered that answer. The Learn guide warns that poorly framed questions still return scores, and its best practices suggest adding an abstention option such as "cannot tell" when the model shouldn't make a forced choice.

Our hypothesis is that one new option, `unclear`, with a clear description, will give vague messages somewhere to go.

## 10. Fix it and verify

We add `unclear` to the options and run the same 13 messages again. Nothing else changes: same state, same questions, same descriptions for the seven teams.

In [10]:
TEAMS_WITH_ABSTAIN = {
    **TEAMS,
    "unclear": "Not enough information to choose a team, or the message does not describe a specific problem",
}

after = run_batch(STARTER, TEAMS_WITH_ABSTAIN)

print(f"Before: {before.match.mean():.0%} of {len(before)} messages routed to the expected team")
print(f"After : {after.match.mean():.0%} of {len(after)} messages routed to the expected team\n")

compare = before[["id", "message", "expected", "team", "conf"]].merge(
    after[["id", "team", "conf"]], on="id", suffixes=("_before", "_after")
)
compare[compare.expected == "unclear"]

Before: 62% of 13 messages routed to the expected team
After : 100% of 13 messages routed to the expected team



,id,message,expected,team_before,conf_before,team_after,conf_after
8,m27,It's happening again over there.,unclear,security,0.92,unclear,0.98
9,m34,Help,unclear,security,0.62,unclear,0.99
10,m43,Can someone come?,unclear,guest_services,0.40,unclear,0.99
11,m56,Is it safe?,unclear,command,0.84,unclear,0.94
12,m69,Somebody should take a look at the thing nea,unclear,security,0.91,unclear,0.98


### 10.1 Why this works

An abstention option changes what a low probability means. Before, a vague message had to put its probability on a wrong team. Now it can put it on `unclear`, and our code treats that as "send this to a person." It's also a deliberate design choice. We decide what the model may answer, so we also decide what counts as an answer the desk can't act on.

The clear messages should look the same as before. If one changed, the new option might overlap with a team description, and that's worth checking in the table.

## 11. Your Turn to Explore

1. Shorten every description in `TEAMS` to a single word, such as `"health"` or `"food"`, and rerun the batch. How do accuracy and `conf` change? Wording matters, and this makes it visible.
2. Try the Spanish and French messages, `m17` and `m36`, and a few of your own. Does the severity and danger signal hold up in another language?
3. Shuffle the order of the options in `TEAMS_WITH_ABSTAIN` and rerun. The Learn guide suggests testing whether order changes the answer. Does it?

In [11]:
# Your experiments here

## 12. Summary

We used Microsoft-Decision-1 on the Contoso Music Festival ops desk to:

- Ask choice, score, and noul questions about one message in a single request
- Pass JSON, including open incidents, as the state
- Read the answers as probabilities and flatten them into a table
- Find out why vague messages got a confident team, and fix it with an `unclear` option

Reach for a decision model when a step has a fixed set of answers and the application needs a typed result it can branch on. Reach for a generative model when the application needs words, such as a reply, a summary, or an explanation. Notebook 02 asks the next question: when is a probability confident enough to act on? For background, see the [decision models primer](../../../docs/primers/decision-models.md) and the [model cards primer](../../../docs/primers/model-cards.md).

**Next steps**

- [02-set-confidence-thresholds.ipynb](02-set-confidence-thresholds.ipynb) - check calibration and sort decisions into lanes
- [03-guard-and-grade-the-ops-copilot.ipynb](03-guard-and-grade-the-ops-copilot.ipynb) - gate agent actions and grade replies

## 13. References

- [Deploy and use Microsoft-Decision-1 in Microsoft Foundry](https://learn.microsoft.com/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision) - request and response shapes, question types, limitations, and best practices
- [Microsoft-Decision-1 model card](https://ai.azure.com/catalog/models/Microsoft-Decision-1) - capabilities, intended uses, and out-of-scope uses
- [Introducing Microsoft-Decision-1 in Microsoft Foundry](https://techcommunity.microsoft.com/blog/azure-ai-foundry-blog/introducing-microsoft-decision-1-in-microsoft-foundry-for-decision-and-classific/4562742) - release context
- [Decision models primer](../../../docs/primers/decision-models.md) - what a decision API is